# E8-v2 — Colab CPU validation and execution

This notebook runs the **pretrained FastText Common Crawl `cc.en.300.bin`** retrieval variant. It does not train or fine-tune FastText. The original E8 experiment is not modified.

**Important:** the full binary is multi-GB. The notebook checks free disk and RAM before download. It streams the official gzip artifact to temporary runtime storage, loads it once, and does not place it in Git or Google Drive. The model remains in memory during retrieval.

ToolRet and MCPToolBench++ are evaluated as separate tracks. Primary metrics remain Hit@1, Hit@5, MRR@10 and Macro Hit@1.

**Dataset compatibility:** this notebook now reuses the dataset layout from the previous MultiBenchmark Colab run under `/content/taas-provenance-discovery/data`. It does not require a Google Drive dataset copy. Colab runtimes are ephemeral: if the previous runtime was reset, rerun the original pinned acquisition/preprocessing workflow before this benchmark.

In [ ]:
import os, sys, json, shutil, subprocess, pathlib, platform, time, gc
from pathlib import Path
print('Python:', sys.version)
print('Platform:', platform.platform())
print('CPU cores:', os.cpu_count())
print('Disk free GiB:', round(shutil.disk_usage('/content').free / 1024**3, 2))
try:
    import psutil
    vm=psutil.virtual_memory()
    print('RAM total GiB:', round(vm.total/1024**3,2), '| available GiB:', round(vm.available/1024**3,2))
except ImportError:
    print('psutil not installed yet')


## 1. Clone the experiment branch and reuse the previous Colab dataset layout

The previous MultiBenchmark workflow stored datasets inside the repository checkout:
- ToolRet: `data/processed/toolret`
- MCPToolBench++: `data/raw`

The notebook checks out the exact E8-v2 branch first, then validates these paths. It deliberately does not silently download or rebuild benchmark data.

In [ ]:
# Reuse an existing checkout without deleting its untracked dataset files.
if not REPO_DIR.exists():
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO_URL,str(REPO_DIR)],check=True)
else:
    origin = subprocess.check_output(
        ['git','-C',str(REPO_DIR),'remote','get-url','origin'],text=True
    ).strip().rstrip('/')
    if origin not in {REPO_URL, REPO_URL.removesuffix('.git')}:
        raise RuntimeError(f'Unexpected repository origin at {REPO_DIR}: {origin}')
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','--depth','1','origin',BRANCH],check=True)
    subprocess.run(
        ['git','-C',str(REPO_DIR),'checkout','-B',BRANCH,'FETCH_HEAD'],check=True
    )


## 2. Install CPU dependencies in the checked-out experiment branch

The notebook uses the existing checkout and never deletes it, because it contains the datasets acquired by the previous Colab workflow. No benchmark dataset or model artifact is committed.

In [ ]:
%cd /content/taas-provenance-discovery
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.','pytest','fasttext-wheel==0.9.2'],check=True)
print('Checked out:', subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
print('Branch:', subprocess.check_output(['git','branch','--show-current'],text=True).strip())

## 3. Run unit tests and syntax/config checks

These checks validate the implementation mechanics only; they do not establish retrieval quality or prove that the real pretrained model loads successfully.

In [ ]:
subprocess.run([sys.executable,'-m','pytest','-q','tests/test_e8_cpu_static_embedding.py'],check=True)
subprocess.run([sys.executable,'-m','py_compile','experiments/e8_cpu_static_embedding.py'],check=True)
subprocess.run([sys.executable,'-m','json.tool','docs/experiment_design/E8_CPU_STATIC_EMBEDDING_v2.json'],stdout=subprocess.DEVNULL,check=True)
print('Unit tests, Python compilation and JSON validation passed.')


## 4. Resource gate and load pretrained model (inference only)

The official Common Crawl model is large. The 8 GiB free-disk threshold is a safety gate, not a guarantee that RAM is sufficient. If available RAM is below 10 GiB, stop and select a larger Colab runtime rather than risking an OOM. Do not switch to a different model silently.

In [ ]:
## 6. Run 100-query smoke tests independently

Run this gate first. It uses the complete candidate tool universe but only the first 100 discovery queries per track. Smoke-test artifacts are written under `/content/e8-results`. Colab runtime storage is temporary; copy the results to Drive after the run if persistence is required.

## 5. Load both benchmark tracks and inspect denominators

The evaluator labels are consumed only by the metric evaluator after rankings have been generated. The IDF model is fitted only over candidate tool documents, independently per track.

In [ ]:
## 7. Full benchmark — run only after reviewing smoke results

Set `RUN_FULL = True` only after checking the smoke outputs, memory headroom, dataset manifests and expected runtime. This runs each dataset track separately and saves rankings/checkpoints/metrics under `/content/e8-results`.

## 6. Run 100-query smoke tests independently

Run this gate first. It uses the complete candidate tool universe but only the first 100 discovery queries per track. It writes smoke-test artifacts to Drive, separate from the full-run directory.

In [ ]:
## 8. Export a compact validation record

This record includes the exact Git commit, model hash, runtime telemetry, smoke results and full-run summary if executed. It does not include the model binary. Results are stored in the temporary Colab runtime at `/content/e8-results`; copy this directory to Drive if you need persistence.

## 7. Full benchmark — run only after reviewing smoke results

Set `RUN_FULL = True` only after checking the smoke outputs, memory headroom, dataset manifests and expected runtime. This runs each dataset track separately and saves rankings/checkpoints/metrics to Drive.

In [ ]:
RUN_FULL = False  # Explicit opt-in after smoke-test review
if RUN_FULL:
    FULL_DIR = RESULTS_DIR/'full'
    FULL_DIR.mkdir(parents=True,exist_ok=True)
    full_summaries=[]
    for track in (toolret, mcpt):
        out=FULL_DIR/track.dataset_id
        result=run_track(track,fasttext_model,out,model_sha256,{'top_k':10,'checkpoint_tasks':16})
        summary={'track':track.dataset_id,'tasks':len(track.tasks),'candidate_tools':len(track.tools),
                 'primary_metrics':result['metrics_primary_qrels'],
                 'secondary_sensitivity_metrics':result['metrics_secondary_sensitivity_qrels'],
                 'timing_seconds':result['timing_seconds']}
        full_summaries.append(summary)
        print(json.dumps(summary,indent=2))
    (FULL_DIR/'FULL_SUMMARY.json').write_text(json.dumps(full_summaries,indent=2)+'\n')
else:
    print('Full run is disabled. Set RUN_FULL=True only after smoke validation.')


## 8. Export a compact validation record

This record includes the exact Git commit, model hash, runtime telemetry, smoke results and full-run summary if executed. It does not include the model binary.

In [ ]:
validation_record={
 'experiment':'E8-CPU-StaticEmbedding-v2','training_performed':False,
 'git_commit':subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip(),
 'branch':BRANCH,'model_sha256':model_sha256,'model_metadata':model_metadata,
 'model_load_seconds':load_seconds,'runtime':{'python':sys.version,'platform':platform.platform(),
             'cpu_count':os.cpu_count(),'ram_available_gib_after_load':round(psutil.virtual_memory().available/1024**3,2)},
 'smoke_summary':smoke_summaries,'full_run_enabled':RUN_FULL}
(RESULTS_DIR/'COLAB_VALIDATION_RECORD.json').write_text(json.dumps(validation_record,indent=2)+'\n')
print('Validation record:', RESULTS_DIR/'COLAB_VALIDATION_RECORD.json')
print(json.dumps(validation_record,indent=2))


## Scientific and operational cautions

- The smoke test is a pipeline check, not a statistically powered comparison.
- Do not pool ToolRet and MCPToolBench++ metrics.
- Do not tune IDF, preprocessing, or thresholds on evaluator qrels.
- The model hash, dataset manifests, query/tool counts and exclusions must accompany any reported result.
- The notebook does not merge the PR or alter the original E8 run.

Official model documentation: https://fasttext.cc/docs/en/crawl-vectors
